In [ ]:
# prompt: connect the drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from skimage.io import imread, imsave
from skimage.util import img_as_float
from skimage.transform import resize
from skimage.io import imshow
from scipy.spatial.distance import directed_hausdorff, cdist  # Import de cdist ajouté
from matplotlib import pyplot as plt
import tensorflow.keras.backend as K
import sys
sys.path.append('/content/drive/MyDrive/ProjetDL2024')
import model
import metrics_and_losses

DATA_PATH = '/content/drive/MyDrive/ProjetDL2024/Dataset'

TRAIN_FRAME_PATH = DATA_PATH + '/frames/Train'
TRAIN_MASK_PATH = DATA_PATH + '/masks endo/Train'

VAL_FRAME_PATH = DATA_PATH + '/frames/Val'
VAL_MASK_PATH = DATA_PATH + '/masks endo/Val'

TEST_FRAME_PATH = DATA_PATH + '/frames/Test'
TEST_MASK_PATH = DATA_PATH + '/masks endo/Test'

WEIGHTS_PATH = '/content/drive/MyDrive/ProjetDL2024'

IMG_WIDTH = 256
IMG_HEIGHT = 256

# Load model with optimized weights
m = model.get_UNet(img_rows=256, img_cols=256)

# Load weights
m.load_weights(WEIGHTS_PATH + '/My_weights.weights.h5')


In [ ]:
# Build a 4D numpy matrix to store all training images from TRAIN_FRAME_PATH
# output = img_train with indices 0 for image number, 1 ordinate, 2 abscissa, 3 channel (0 for gray level images)

IMG_WIDTH = 256
IMG_HEIGHT = 256

train_frames = os.listdir(TRAIN_FRAME_PATH)
img_train = np.zeros((len(train_frames), IMG_HEIGHT, IMG_WIDTH, 1))

n = 0
for frame in train_frames:
    img = imread(TRAIN_FRAME_PATH + '/' + frame)
    img = resize(img, (IMG_HEIGHT, IMG_WIDTH))
    img_train[n, :, :, 0] = img
    n = n + 1

# Build a 4D numpy matrix to store all validation images from VAL_FRAME_PATH
# output = img_val with indices 0 for image number, 1 ordinate, 2 abscissa, 3 channel (0 for gray level images)
val_frames = os.listdir(VAL_FRAME_PATH)
img_val = np.zeros((len(val_frames), IMG_HEIGHT, IMG_WIDTH, 1))

n = 0
for frame in val_frames:
    img = imread(VAL_FRAME_PATH + '/' + frame)
    img = resize(img, (IMG_HEIGHT, IMG_WIDTH))
    img_val[n, :, :, 0] = img
    n = n + 1



In [ ]:

IMG_HEIGHT, IMG_WIDTH = 256, 256

# Fusion between initial images and predicted masks for display purpose
display_train = np.zeros((len(train_frames), IMG_HEIGHT, IMG_WIDTH, 3))
display_val = np.zeros((len(val_frames), IMG_HEIGHT, IMG_WIDTH, 3))

# Paths to save predicted images
TRAIN_PRED_PATH = DATA_PATH + '/train_preds'
if not os.path.isdir(TRAIN_PRED_PATH):
    os.makedirs(TRAIN_PRED_PATH)
VAL_PRED_PATH = DATA_PATH + '/val_preds'
if not os.path.isdir(VAL_PRED_PATH):
    os.makedirs(VAL_PRED_PATH)

# Charger et prétraiter les images
def preprocess_images(frame_path, frame_list):
    images = []
    for frame in frame_list:
        img = imread(os.path.join(frame_path, frame), as_gray=True)  # Charger en niveaux de gris
        img = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)  # Redimensionner
        images.append(img / 255.0)  # Normaliser entre 0 et 1
    return np.expand_dims(np.array(images), axis=-1)  # Ajouter un axe pour le canal

# Charger les masks
def preprocess_masks(mask_path, mask_list):
    masks = []
    for mask in mask_list:
        img = imread(os.path.join(mask_path, mask), as_gray=True)  # Charger en niveaux de gris
        img = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)  # Redimensionner
        masks.append(img / 255.0)  # Normaliser entre 0 et 1
    return np.expand_dims(np.array(masks), axis=-1)  # Ajouter un axe pour le canal

 # Paths to save masks
train_masks = preprocess_masks(TRAIN_MASK_PATH, os.listdir(TRAIN_MASK_PATH))
val_masks = preprocess_masks(VAL_MASK_PATH, os.listdir(VAL_MASK_PATH))


# Charger les données d'entraînement et de validation
train_frames = preprocess_images(TRAIN_FRAME_PATH, os.listdir(TRAIN_FRAME_PATH))
val_frames = preprocess_images(VAL_FRAME_PATH, os.listdir(VAL_FRAME_PATH))

# Save the fusion of training images and corresponding predicted masks to TRAIN_PRED_PATH
for i, frame in enumerate(train_frames):
    predicted_mask = m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0]  # Prédire le masque
    predicted_mask_binary = (predicted_mask > 0.5).astype(np.uint8)  # Appliquer un seuil binaire
    fused_image = np.stack((frame[:, :, 0], predicted_mask_binary * 255, frame[:, :, 0]), axis=-1)
    display_train[i] = fused_image
    save_path = os.path.join(TRAIN_PRED_PATH, f"train_fusion_{i}.png")
    imsave(save_path, fused_image.astype(np.uint8))

# Save the fusion of validation images and corresponding predicted masks to VAL_PRED_PATH
for i, frame in enumerate(val_frames):
    predicted_mask = m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0]
    predicted_mask_binary = (predicted_mask > 0.5).astype(np.uint8)
    fused_image = np.stack((frame[:, :, 0], predicted_mask_binary * 255, frame[:, :, 0]), axis=-1)
    display_val[i] = fused_image
    save_path = os.path.join(VAL_PRED_PATH, f"val_fusion_{i}.png")
    imsave(save_path, fused_image.astype(np.uint8))

# Compute the average Dice coefficient for the training dataset
def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

train_dice_scores = []
for i, frame in enumerate(train_frames):
    predicted_mask = m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0]
    predicted_mask_binary = (predicted_mask > 0.5).astype(np.uint8)
    train_dice_scores.append(dice_coefficient(train_masks[i, :, :, 0], predicted_mask_binary))
mean_dice_train = np.mean(train_dice_scores)

# Compute the average Dice coefficient for the validation dataset
val_dice_scores = []
for i, frame in enumerate(val_frames):
    predicted_mask = m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0]
    predicted_mask_binary = (predicted_mask > 0.5).astype(np.uint8)
    val_dice_scores.append(dice_coefficient(val_masks[i, :, :, 0], predicted_mask_binary))

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 3, 1)
    plt.title("Image originale")
    plt.imshow(frame[:, :, 0], cmap="gray")
    plt.subplot(1, 3, 2)
    plt.title("Masque prédit")
    plt.imshow(predicted_mask_binary, cmap="gray")
    plt.subplot(1, 3, 3)
    plt.title("Masque attendu (vérité terrain)")
    plt.imshow(val_masks[i, :, :, 0], cmap="gray")
    plt.show()

mean_dice_val = np.mean(val_dice_scores)

# Compute Euclidean and Haussdorff distances
def euclidean_distance(y_true, y_pred):
    """
    Calculate the mean Euclidean distance between points in the true mask and the predicted mask.
    """
    true_points = np.argwhere(y_true)
    pred_points = np.argwhere(y_pred)
    distances = cdist(true_points, pred_points, metric='euclidean')
    return np.mean(np.min(distances, axis=1))

def hausdorff_distance(y_true, y_pred):
    true_points = np.argwhere(y_true)
    pred_points = np.argwhere(y_pred)
    forward_hd = directed_hausdorff(true_points, pred_points)[0]
    backward_hd = directed_hausdorff(pred_points, true_points)[0]
    return max(forward_hd, backward_hd)

mean_Euclidian_distance = np.mean([euclidean_distance(val_masks[i, :, :, 0], (m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0] > 0.5).astype(np.uint8)) for i, frame in enumerate(val_frames)])
mean_Hausdorff_distance = np.mean([hausdorff_distance(val_masks[i, :, :, 0], (m.predict(np.expand_dims(frame, axis=0))[0, :, :, 0] > 0.5).astype(np.uint8)) for i, frame in enumerate(val_frames)])

print("Mean Dice coefficient for training database = {}".format(mean_dice_train))
print("Mean Dice coefficient for validation database = {}".format(mean_dice_val))
print("Mean Euclidian distance for validation database = {}".format(mean_Euclidian_distance))
print("Mean Haussdorff distance for validation database = {}".format(mean_Hausdorff_distance))

Output hidden; open in https://colab.research.google.com to view.